In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [2]:
df = pd.read_csv("drug_regulatory_classification_dataset.csv")

# Remove missing target
df = df.dropna(subset=["Target_Regulatory_Class"])

In [3]:
X = df.drop("Target_Regulatory_Class", axis=1)
y = df["Target_Regulatory_Class"]

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [5]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

In [6]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler())
])

In [7]:
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

In [8]:
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [9]:
model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

In [10]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model)
])

In [11]:
param_grid = {
    "model__C": [0.01, 0.1, 1, 5, 10, 50]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

Best Parameters: {'model__C': 0.01}


In [12]:
best_model = grid.best_estimator_

In [13]:
y_pred = best_model.predict(X_test)

print("Final Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Final Accuracy: 0.719561403508772
              precision    recall  f1-score   support

           0       0.89      0.71      0.79      8426
           1       0.48      0.75      0.58      2974

    accuracy                           0.72     11400
   macro avg       0.68      0.73      0.69     11400
weighted avg       0.78      0.72      0.73     11400



In [14]:
print("Train Accuracy:", best_model.score(X_train, y_train))
print("Test Accuracy:", best_model.score(X_test, y_test))

Train Accuracy: 0.7218859649122807
Test Accuracy: 0.719561403508772
